# ETM PPD Notebook

Notebook unifie ETM pour reproduire le pipeline datasets + export `.mat` compatible.


## 1) Install (Kaggle-safe)


In [ ]:
!pip install -q --upgrade-strategy only-if-needed scipy pyyaml gensim


## 2) Imports


In [ ]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import scipy
import scipy.io
import scipy.sparse as sp
import torch
from sklearn import metrics

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## 3) Dataset discovery


In [ ]:
TARGET_DATASETS = ['20NG', 'AGNews', 'IMDB', 'YahooAnswer']
K_LIST = [20, 50, 100]
RANDOM_SEED = 42

REQUIRED_DATA_FILES = [
    'train_bow.npz',
    'test_bow.npz',
    'train_labels.txt',
    'test_labels.txt',
    'vocab.txt',
    'word_embeddings.npz',
]

IS_KAGGLE = Path('/kaggle').exists()


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ECTRM_source').exists():
            return candidate
    return start


if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working')
    INPUT_ROOT = Path('/kaggle/input')
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    INPUT_ROOT = PROJECT_ROOT


def has_required_files(path: Path) -> bool:
    return path.exists() and path.is_dir() and all((path / f).exists() for f in REQUIRED_DATA_FILES)


def guess_dataset_name(path: Path):
    s = str(path).lower()
    if '20ng' in s or '20news' in s:
        return '20NG'
    if 'agnews' in s or 'ag_news' in s:
        return 'AGNews'
    if 'yahoo' in s:
        return 'YahooAnswer'
    if 'imdb' in s:
        return 'IMDB'
    return None


def iter_candidate_dirs(base: Path, max_depth: int = 5):
    if not base.exists():
        return
    base_depth = len(base.parts)
    for root, dirs, _ in os.walk(base):
        root_path = Path(root)
        depth = len(root_path.parts) - base_depth
        if depth > max_depth:
            dirs[:] = []
            continue
        yield root_path


def discover_dataset_dirs():
    found = {}

    local_roots = [
        PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'data',
        PROJECT_ROOT / 'ECTRM_source' / 'data',
    ]

    for root in local_roots:
        for ds in TARGET_DATASETS:
            p = root / ds
            if has_required_files(p):
                found[ds] = p

    scan_roots = [Path('/kaggle/input'), PROJECT_ROOT, INPUT_ROOT] if IS_KAGGLE else [PROJECT_ROOT, INPUT_ROOT]
    seen = set()
    for scan_root in scan_roots:
        for cand in iter_candidate_dirs(scan_root):
            cs = str(cand)
            if cs in seen:
                continue
            seen.add(cs)
            if not has_required_files(cand):
                continue
            ds = guess_dataset_name(cand)
            if ds is not None and ds not in found:
                found[ds] = cand

    return found


DATASET_DIRS = discover_dataset_dirs()
print('IS_KAGGLE:', IS_KAGGLE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('INPUT_ROOT:', INPUT_ROOT)
print('\\nResolved datasets:')
for ds in TARGET_DATASETS:
    print(' -', ds, DATASET_DIRS.get(ds))


## 4) Load official ETM source


In [ ]:
def looks_like_etm_repo(path: Path) -> bool:
    return path.exists() and (path / 'etm.py').exists() and (path / 'main.py').exists()


candidate_etm_repos = [
    PROJECT_ROOT / 'ETM_source' / 'ETM',
    PROJECT_ROOT / 'ETM_source',
]

etm_repo = None
for cand in candidate_etm_repos:
    if looks_like_etm_repo(cand):
        etm_repo = cand
        break

if etm_repo is None:
    for base in [Path('/kaggle/input'), Path('/kaggle/working'), PROJECT_ROOT]:
        if not base.exists():
            continue
        for r, _, files in os.walk(base):
            rp = Path(r)
            if 'etm.py' in files and 'main.py' in files:
                if looks_like_etm_repo(rp):
                    etm_repo = rp
                    break
        if etm_repo is not None:
            break

if etm_repo is None:
    raise FileNotFoundError('Impossible de localiser ETM source (etm.py + main.py)')

sys.path.insert(0, str(etm_repo))
from etm import ETM

print('ETM repo:', etm_repo)


## 5) Training + export


In [ ]:
OUTPUT_DIR = (Path('/kaggle/working') / 'Sortie_mat' / 'output' / 'kaggle' / 'ETM') if IS_KAGGLE else (PROJECT_ROOT / 'Sortie_mat' / 'ETM')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)

ETM_DEFAULTS = {
    'epochs': 100,
    'epochs_20NG': 150,
    'batch_size': 1000,
    'lr': 5e-3,
    'weight_decay': 1.2e-6,
    't_hidden_size': 800,
    'theta_act': 'relu',
    'enc_drop': 0.0,
    'train_embeddings': False,
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_vocab(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]


def load_bow_csr(path: Path):
    return sp.load_npz(path).astype(np.float32).tocsr()


def load_word_embeddings(path: Path, vocab_size=None):
    try:
        arr = sp.load_npz(path).astype(np.float32).toarray()
        if vocab_size is not None and arr.shape[0] != vocab_size and arr.shape[1] == vocab_size:
            arr = arr.T
        return arr.astype(np.float32)
    except Exception:
        data = np.load(path)
        if isinstance(data, np.lib.npyio.NpzFile):
            arr = data[list(data.keys())[0]]
        else:
            arr = data
        arr = np.asarray(arr, dtype=np.float32)
        if vocab_size is not None:
            if arr.ndim == 1:
                emb_dim = arr.size // vocab_size
                arr = arr[: vocab_size * emb_dim].reshape(vocab_size, emb_dim)
            elif arr.shape[0] != vocab_size and arr.shape[1] == vocab_size:
                arr = arr.T
        return arr.astype(np.float32)


def normalize_rows(x):
    sums = x.sum(1, keepdims=True)
    sums[sums <= 0] = 1.0
    return x / sums


def batch_dense_from_csr(csr_mat, idx):
    return csr_mat[idx].toarray().astype(np.float32)


def build_etm_model(K, vocab_size, word_emb, cfg):
    emb_dim = int(word_emb.shape[1])
    emb_tensor = torch.from_numpy(word_emb)

    model = ETM(
        num_topics=K,
        vocab_size=vocab_size,
        t_hidden_size=int(cfg['t_hidden_size']),
        rho_size=emb_dim,
        emsize=emb_dim,
        theta_act=cfg['theta_act'],
        embeddings=emb_tensor,
        train_embeddings=bool(cfg['train_embeddings']),
        enc_drop=float(cfg['enc_drop']),
    ).to(DEVICE)

    return model


def train_etm(model, train_csr, cfg, seed=42):
    set_seed(seed)
    model.train()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(cfg['lr']),
        weight_decay=float(cfg['weight_decay']),
    )

    N = train_csr.shape[0]
    bsz = int(cfg['batch_size'])

    for epoch in range(int(cfg['epochs'])):
        order = np.random.permutation(N)
        losses = []

        for start in range(0, N, bsz):
            idx = order[start:start + bsz]
            x_np = batch_dense_from_csr(train_csr, idx)
            x_norm_np = normalize_rows(x_np)

            x = torch.from_numpy(x_np).to(DEVICE)
            x_norm = torch.from_numpy(x_norm_np).to(DEVICE)

            recon_loss, kld = model(x, x_norm)
            loss = recon_loss + kld

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            losses.append(float(loss.item()))

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:03d}/{cfg['epochs']} loss={np.mean(losses):.4f}")


def infer_theta_csr(model, csr_mat, batch_size=512):
    model.eval()
    all_theta = []
    N = csr_mat.shape[0]

    with torch.no_grad():
        for start in range(0, N, batch_size):
            idx = np.arange(start, min(start + batch_size, N))
            x_np = batch_dense_from_csr(csr_mat, idx)
            x_norm_np = normalize_rows(x_np)

            x_norm = torch.from_numpy(x_norm_np).to(DEVICE)
            theta, _ = model.get_theta(x_norm)
            all_theta.append(theta.cpu().numpy().astype(np.float32))

    return np.vstack(all_theta)


def run_one_etm(dataset, K, seed=42):
    if dataset not in DATASET_DIRS:
        raise KeyError(f'Dataset {dataset} introuvable. Disponibles: {list(DATASET_DIRS.keys())}')

    cfg = dict(ETM_DEFAULTS)
    if dataset == '20NG':
        cfg['epochs'] = int(cfg.get('epochs_20NG', cfg['epochs']))

    data_dir = DATASET_DIRS[dataset]
    train_csr = load_bow_csr(data_dir / 'train_bow.npz')
    test_csr = load_bow_csr(data_dir / 'test_bow.npz')
    word_emb = load_word_embeddings(data_dir / 'word_embeddings.npz', vocab_size=train_csr.shape[1])

    model = build_etm_model(K, train_csr.shape[1], word_emb, cfg)

    print(f"\\n=== ETM {dataset} K={K} ===")
    train_etm(model, train_csr, cfg, seed=seed)

    beta = model.get_beta().detach().cpu().numpy().astype(np.float32)
    train_theta = infer_theta_csr(model, train_csr, batch_size=int(cfg['batch_size']))
    test_theta = infer_theta_csr(model, test_csr, batch_size=int(cfg['batch_size']))

    ds_out = OUTPUT_DIR / dataset
    ds_out.mkdir(parents=True, exist_ok=True)
    out_path = ds_out / f'{dataset}_ETM_K{K}_seed{seed}.mat'

    scipy.io.savemat(
        str(out_path),
        {
            'beta': beta,
            'train_theta': train_theta,
            'test_theta': test_theta,
            'traintheta': train_theta,
            'testtheta': test_theta,
        },
    )

    print('Saved:', out_path)
    return out_path


## 6) Run


In [ ]:
for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        print('SKIP missing dataset:', dataset)
        continue
    for K in K_LIST:
        run_one_etm(dataset, K=K, seed=RANDOM_SEED)


## 7) Metrics and topics files


In [ ]:
import pandas as pd


def topic_diversity_from_beta(beta, topn=15):
    all_words = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        all_words.extend(top_idx.tolist())
    return len(set(all_words)) / max(1, len(all_words))


def purity_score(y_true, y_pred):
    contingency_matrix = metrics.cluster.contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)


rows = []
TOPN = 15
for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        continue

    ds_out = OUTPUT_DIR / dataset
    vocab = load_vocab(DATASET_DIRS[dataset] / 'vocab.txt')
    labels = np.loadtxt(DATASET_DIRS[dataset] / 'test_labels.txt', dtype=np.int32)

    for K in K_LIST:
        mat_path = ds_out / f'{dataset}_ETM_K{K}_seed{RANDOM_SEED}.mat'
        if not mat_path.exists():
            continue

        loaded = scipy.io.loadmat(str(mat_path))
        beta = loaded['beta']
        test_theta = loaded['test_theta']

        n_eval = min(len(labels), test_theta.shape[0])
        y_true = labels[:n_eval]
        y_pred = np.argmax(test_theta[:n_eval], axis=1)

        rows.append({
            'model': 'ETM',
            'dataset': dataset,
            'K': K,
            'Purity': float(purity_score(y_true, y_pred)),
            'NMI': float(metrics.cluster.normalized_mutual_info_score(y_true, y_pred)),
            'TD_top15': float(topic_diversity_from_beta(beta, topn=15)),
        })

        txt_path = ds_out / f'topics_for_cv_{dataset}_ETM_K{K}_seed{RANDOM_SEED}.txt'
        with open(txt_path, 'w', encoding='utf-8') as f:
            for k in range(beta.shape[0]):
                top_idx = np.argsort(beta[k])[::-1][:TOPN]
                f.write(' '.join(vocab[i] for i in top_idx) + '\n')
        print('Saved:', txt_path)

if rows:
    df = pd.DataFrame(rows).sort_values(['dataset', 'K']).reset_index(drop=True)
    display(df)
    csv_path = OUTPUT_DIR / 'ETM_metrics_TD_Purity_NMI.csv'
    df.to_csv(csv_path, index=False)
    print('Saved:', csv_path)
else:
    print('No ETM .mat found yet')

print('\\nFiles in OUTPUT_DIR:')
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        print('-', p.relative_to(OUTPUT_DIR))
